# SimPipeline quickstart

Runs the full loop: simulate users → stream events → compute features → train the two-tower model → evaluate.

**Locally or in a cloud session:** open this notebook from the repo root after `pip install -e .`
**On Colab:** just run the cells. The setup cell clones the repo for you (it's public, so no token is needed).

Every run with the same `SEED` gives the same events and the same model.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# On Colab, fetch the repo and work inside it. Elsewhere, run from the repo root.
if "google.colab" in sys.modules:
    repo = Path("/content/SimPipelineExp")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/PrimozRavbar/SimPipelineExp.git", str(repo)],
            check=True,
        )
    os.chdir(repo)

ROOT = Path.cwd()
assert (ROOT / "ml_system").is_dir(), f"Open this notebook from the repo root (currently {ROOT})"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Repo:", ROOT)

## Settings

In [ ]:
SEED = 42            # same seed -> same simulation and same model
SIM_TICKS = 5000     # one simulated user session per tick
EPOCHS = 5
NEGATIVE_SAMPLES = 5

## Load MovieLens and build the system

In [ ]:
from world.external_data_loader import load_movielens
from ml_system.build_recommendation_system import build_recommendation_system

world_data = load_movielens()   # uses the ml-100k folder in the repo

system = build_recommendation_system(
    movie_objects=world_data["movie_objects"],
    user_objects=world_data["user_objects"],
    user_states=world_data["user_states"],
    movie_states=world_data["movie_states"],
    seed=SEED,
)

print("Users:", len(world_data["user_objects"]), "| Movies:", len(world_data["movie_objects"]))

## Run the simulation

In [ ]:
from collections import Counter

system["sim"].run(SIM_TICKS)

events = system["datalake"].events
print("DataLake events:", len(events))
print(Counter(type(event).__name__ for event in events))

## Offline features (Spark)

In [ ]:
spark = system["spark"]

spark.run_offline_feature_job(events, system["offline_pipeline"].feature_pipeline)

job = spark.application.jobs[0]
print("Application:", spark.application.state.value, spark.application.metrics)
print("Job:", job.state.value, "| Stage:", job.stages[0].state.value, "| Task:", job.stages[0].tasks[0].state.value)

## Train the two-tower model

In [ ]:
from ml_system.retrain import retrain

system["offline_pipeline"].dataset_generator.negative_samples = NEGATIVE_SAMPLES
system["offline_pipeline"].trainer.epochs = EPOCHS

artifacts = retrain(system)

## Recall@20 on the training events

This scores the model on the same events it trained on, so it measures fit, not generalization. A held-out evaluation with popularity and random baselines comes in the next phase.

In [ ]:
from simulation.events import RecommendationClickedEvent, WatchEvent, RetrievalRequest

K = 20
hits = evaluated = 0

for event in events:
    if not isinstance(event, (RecommendationClickedEvent, WatchEvent)):
        continue

    recommendations = system["retriever"].retrieve(RetrievalRequest(user_id=event.user_id, k=K))
    evaluated += 1
    hits += event.movie_id in recommendations

print(f"Recall@{K} (in-sample): {hits / max(evaluated, 1):.2%}  ({hits}/{evaluated})")